In [1]:
import json

import boto3
import s3fs
import xarray as xr

- Make a new bucket
- Add the proper policy
- Write a zarr file to the bucket
- ingest that zarr file to SH

In [2]:
# Make new Bucket
region = "eu-central-1"
bucket_name = "sh-change-detection-test"

In [ ]:
s3_client = boto3.client("s3", region_name=region)
location = {"LocationConstraint": region}
s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration=location)

In [3]:
# Set SH BYOC bucket policy
bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "Sentinel Hub permissions",
            "Effect": "Allow",
            "Principal": {"AWS": "arn:aws:iam::614251495211:root"},
            "Action": ["s3:GetBucketLocation", "s3:ListBucket", "s3:GetObject"],
            "Resource": [f"arn:aws:s3:::{bucket_name}", f"arn:aws:s3:::{bucket_name}/*"],
        }
    ],
}

# Convert the policy from JSON dict to string
bucket_policy = json.dumps(bucket_policy)

# Set the new policy
s3 = boto3.client("s3")
s3.put_bucket_policy(Bucket=bucket_name, Policy=bucket_policy)

{'ResponseMetadata': {'RequestId': '35SA286GNXZAXJ34',
  'HostId': 'yk79HaCo1TFgnMoqiDsbHf72EiGXXXhyaF7jqbTHDJ0d6PKjHgdXaEGrt1EAIPTx1pS/x3pu3U56+0NtIw5l5w==',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': 'yk79HaCo1TFgnMoqiDsbHf72EiGXXXhyaF7jqbTHDJ0d6PKjHgdXaEGrt1EAIPTx1pS/x3pu3U56+0NtIw5l5w==',
   'x-amz-request-id': '35SA286GNXZAXJ34',
   'date': 'Fri, 05 Jan 2024 14:16:52 GMT',
   'server': 'AmazonS3'},
  'RetryAttempts': 0}}

In [3]:
test = xr.open_dataset("example.tiff", band_as_variable=True)

In [4]:
test = test.assign_coords({"time": [1704466677]})
for da in test:
    test[da] = test[da].expand_dims({"time": 1})

In [5]:
test["time"] = test.time.assign_attrs({"units": "seconds since 1970-01-01 00:00:00"})

In [6]:
order = ["time", "y", "x"]
reordered_indexes = {index_name: test.indexes[index_name] for index_name in order}
test = test.reindex(reordered_indexes)

In [20]:
test

<xarray.Dataset>
Dimensions:      (x: 512, y: 343, time: 1)
Coordinates:
  * x            (x) float64 12.45 12.45 12.45 12.45 ... 12.54 12.54 12.54 12.54
  * y            (y) float64 41.92 41.92 41.92 41.92 ... 41.87 41.87 41.87 41.87
    spatial_ref  int64 ...
  * time         (time) int64 1704466677
Data variables:
    band_1       (time, y, x) float32 139.0 142.0 143.0 ... 255.0 255.0 255.0
    band_2       (time, y, x) float32 149.0 148.0 146.0 ... 255.0 255.0 255.0
    band_3       (time, y, x) float32 146.0 147.0 142.0 ... 255.0 255.0 255.0
Attributes:
    AREA_OR_POINT:           Area
    TIFFTAG_RESOLUTIONUNIT:  1 (unitless)
    TIFFTAG_XRESOLUTION:     1
    TIFFTAG_YRESOLUTION:     1

In [21]:
test.to_zarr("test_zarr.zarr", mode="w")

/home/jviehweger/Documents/Projects/2024/change-detection/.venv/lib/python3.10/site-packages/xarray/core/dataset.py:2528: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  return to_zarr(  # type: ignore[call-overload,misc]


In [8]:
test["band_1"] = test["band_1"] / 2

In [9]:
s3_out = s3fs.S3FileSystem(anon=False)
store_out = s3fs.S3Map(root=f"s3://{bucket_name}/ds_s3.zarr", s3=s3_out, check=False)
test.to_zarr(store_out, mode="a")

/home/jviehweger/Documents/Projects/2024/change-detection/.venv/lib/python3.10/site-packages/xarray/core/dataset.py:2528: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  return to_zarr(  # type: ignore[call-overload,misc]
